# PBB-style JEPA on CIFAR-10 (40% sparse pool) — standalone

Direct port of the working PBB v2 recipe. The whole folder (`pbb_core.py` + this notebook + the DVS notebook) is copy-paste portable — no `omnibird/` dependency.

**Architecture (one path):**
```
40% sampled pixels (~410 per image, (y, x, R, G, B))
        ↓
Tokenizer:  signal_proj(RGB) + pos_proj(γ_Fourier(y, x))
        ↓
BigBird encoder (6 layers, random SFC permutation per layer)
        ↓
per-pixel features  (B, K_pool, D)
        ↓
gather at TARGET pixel positions     gather at CONTEXT pixel positions
        ↓                                       ↓
target_center → per-token LN              predictor input
        ↓                                       ↓
h_tgt                                  predictor (dense Transformer,
                                       mask-tokens at target coords,
                                       pos_symmetric=True)
                                            ↓
                                          h_pred
        ↓______________ smooth-L1 ___________↓
                       loss = smooth_L1(h_pred, h_tgt)
                       EMA target (stop-grad)
```

**Why this works on CIFAR-10 where the heavier xattn was overkill:** each pixel token carries 24 bits of RGB content. The encoder has rich per-token input, so direct per-pixel target gather (PBB's recipe) provides 12.5× more supervision than the patch-level cross-attn pool — and the encoder doesn't need the architectural defenses (LocalCrossAttention, tokenizer skip, symmetric pool) that we built for 1-bit event tokens.

**Recipe:**
- Per-pixel tokens (no patchification).
- BigBird encoder with multi-curve serialization per layer.
- Multi-block KNN-disjoint target sampling: 4 blocks × 50 pixels = 200 targets per sample.
- Direct per-pixel target gather from the full-pool encoder output.
- DINO-style target centering + per-token LayerNorm before the smooth-L1 loss.
- Dense Transformer predictor with mask-tokens at target coords (`pos_symmetric=True`).
- EMA target, no other regularizer.


## 1. Setup

In [ ]:
import os, sys, math, time, copy, ssl
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
sys.path.insert(0, os.path.abspath('.'))   # so pbb_core.py is importable
ssl._create_default_https_context = ssl._create_unverified_context

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import torchvision

# Standalone library (sits in the same folder as this notebook)
from pbb_core import (
    PBBEncoder, PBBPredictor,
    TargetCenter, ema_update, make_momentum_schedule,
    jepa_loss, gather_target_features,
    precompute_grid_orderings, short_params,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0); np.random.seed(0)

N_GPUS_REQUESTED = 4
N_GPUS = min(N_GPUS_REQUESTED, torch.cuda.device_count())
USE_DP = (DEVICE == "cuda") and (N_GPUS > 1)
print(f"GPUs visible = {torch.cuda.device_count()}  using = {N_GPUS}  DataParallel = {USE_DP}")

# ── Config ───────────────────────────────────────────────────────────────
IMAGE_SIZE = 32
FRAC_POOL  = 0.4
K_POOL     = int(round(FRAC_POOL * IMAGE_SIZE * IMAGE_SIZE))  # 410
K_CTX      = 100
N_PRED     = 4
K_TGT_PER_BLOCK = 50
N_TGT      = N_PRED * K_TGT_PER_BLOCK                          # 200

D_MODEL      = 256
N_LAYERS_ENC = 6
N_HEADS      = 8
DIM_HEAD     = 32
BLOCK_SIZE   = 8
WINDOW       = 1
N_RANDOM     = 2
N_GLOBAL     = 2
FOURIER_DIM  = 96
FOURIER_SCALE = 15.0

D_PRED        = 192
N_LAYERS_PRED = 4
N_HEADS_PRED  = 6
DIM_HEAD_PRED = 32

EMA_START = 0.999
EMA_END   = 1.0

EPOCHS         = 1000
BATCH_SIZE     = 64
LR             = 2e-4
WEIGHT_DECAY   = 0.0
WARMUP_EPOCHS  = 5
PROBE_INTERVAL = 10
PROBE_EPOCHS   = 2
LOG_EVERY      = 50
CKPT_DIR       = "./checkpoints_pbb_cifar10"
os.makedirs(CKPT_DIR, exist_ok=True)
print(f"K_pool={K_POOL}  K_ctx={K_CTX}  N_tgt={N_TGT}  ({N_PRED}x{K_TGT_PER_BLOCK})")


## 2. CIFAR-10 sparse-pool dataset (i-JEPA-faithful multi-block masking)

For each image:
1. Sample `K_POOL = 410` pixels (40%) with a per-image fixed permutation.
2. Sort pool by 2-D Hilbert curve.
3. `train=True`: pick 4 target anchors uniformly in the pool; each anchor's K-nearest pool members form one target block. Context = sample one anchor in the **remaining** pool and take its K-nearest pool members (disjoint from all targets).
4. `train=False`: just expose the first `K_POOL // 2` pixels as context (20% test-time input); no targets.


In [ ]:
# Precompute Hilbert ranks for the 32x32 grid (used to sort the pool).
GRID_RANKS = {k: v for k, v in precompute_grid_orderings(IMAGE_SIZE, ndim=2).items()}

class JEPAChunkCIFAR10(Dataset):
    def __init__(self, base, train=True, pool_seed=0):
        self.base = base
        self.train = train
        self.N_pix = IMAGE_SIZE * IMAGE_SIZE
        rng = np.random.RandomState(pool_seed)
        self.pool_idx = np.stack(
            [rng.permutation(self.N_pix)[:K_POOL] for _ in range(len(base))],
            axis=0,
        ).astype(np.int64)
        ys, xs = torch.meshgrid(
            torch.linspace(-1.0, 1.0, IMAGE_SIZE),
            torch.linspace(-1.0, 1.0, IMAGE_SIZE),
            indexing='ij',
        )
        self.coords_all = torch.stack([ys, xs], dim=-1).view(self.N_pix, 2).float()

    def __len__(self): return len(self.base)

    def _knn_block(self, pool_xy, candidate_local, K, anchor_local, exclude_mask):
        # Returns K local pool indices: anchor + K-1 nearest within `candidate_local`
        d2 = ((pool_xy - pool_xy[anchor_local]) ** 2).sum(-1)
        d2[~candidate_local] = float("inf")
        d2[exclude_mask]     = float("inf")
        topk = torch.topk(d2, K, largest=False).indices
        return topk

    def __getitem__(self, idx):
        img, label = self.base[idx]
        if isinstance(img, torch.Tensor):
            img_t = img.float()
        else:
            img_t = torch.from_numpy(np.array(img)).permute(2, 0, 1).float() / 255.0
        img_t = img_t * 2.0 - 1.0
        rgb = img_t.permute(1, 2, 0).reshape(-1, 3)
        pool_idx_np = self.pool_idx[idx]
        pool_coords = self.coords_all[pool_idx_np]
        pool_signal = rgb[pool_idx_np]
        # Hilbert sort
        full_ranks = GRID_RANKS["hilbert"]
        pool_ranks = full_ranks[pool_idx_np]
        order = pool_ranks.argsort()
        pool_coords = pool_coords[order]
        pool_signal = pool_signal[order]

        K_p = pool_coords.size(0)
        candidate = torch.ones(K_p, dtype=torch.bool)
        exclude   = torch.zeros(K_p, dtype=torch.bool)

        if self.train:
            rng = np.random.RandomState()
            tgt_blocks = []
            for _ in range(N_PRED):
                allowed = (candidate & ~exclude).nonzero(as_tuple=False).squeeze(-1)
                if len(allowed) < K_TGT_PER_BLOCK:
                    break
                anchor = allowed[rng.randint(len(allowed))].item()
                blk = self._knn_block(pool_coords, candidate, K_TGT_PER_BLOCK,
                                       anchor, exclude)
                tgt_blocks.append(blk)
                exclude[blk] = True
            tgt_pool_pos = torch.cat(tgt_blocks) if tgt_blocks else torch.zeros(0, dtype=torch.long)
            expected = N_PRED * K_TGT_PER_BLOCK
            if len(tgt_pool_pos) < expected:
                pad = torch.full((expected - len(tgt_pool_pos),),
                                  tgt_pool_pos[-1].item() if len(tgt_pool_pos) else 0,
                                  dtype=torch.long)
                tgt_pool_pos = torch.cat([tgt_pool_pos, pad])

            # Context: anchor in remaining; KNN in candidate \ exclude
            remaining = (candidate & ~exclude).nonzero(as_tuple=False).squeeze(-1)
            if len(remaining) >= K_CTX:
                anchor = remaining[rng.randint(len(remaining))].item()
                ctx_pool_pos = self._knn_block(pool_coords, candidate, K_CTX, anchor, exclude)
            else:
                ctx_pool_pos = remaining
                if len(ctx_pool_pos) < K_CTX:
                    pad = torch.full((K_CTX - len(ctx_pool_pos),),
                                      ctx_pool_pos[0].item() if len(ctx_pool_pos) else 0,
                                      dtype=torch.long)
                    ctx_pool_pos = torch.cat([ctx_pool_pos, pad])

            return {
                "pool_coords":   pool_coords.contiguous(),
                "pool_signal":   pool_signal.contiguous(),
                "tgt_pool_pos":  tgt_pool_pos.contiguous(),
                "ctx_pool_pos":  ctx_pool_pos.contiguous(),
                "label":         int(label),
            }
        else:
            # Test: first K_POOL//2 of pool = context (20% test input); no targets.
            ctx_pool_pos = torch.arange(K_p // 2, dtype=torch.long)
            return {
                "pool_coords":   pool_coords.contiguous(),
                "pool_signal":   pool_signal.contiguous(),
                "ctx_pool_pos":  ctx_pool_pos.contiguous(),
                "label":         int(label),
            }


CIFAR_ROOT = os.path.expanduser("~/data/cifar10")
os.makedirs(CIFAR_ROOT, exist_ok=True)
cifar_train = torchvision.datasets.CIFAR10(root=CIFAR_ROOT, train=True,  download=True)
cifar_test  = torchvision.datasets.CIFAR10(root=CIFAR_ROOT, train=False, download=True)
train_ds      = JEPAChunkCIFAR10(cifar_train, train=True,  pool_seed=0)
train_eval_ds = JEPAChunkCIFAR10(cifar_train, train=False, pool_seed=0)
test_ds       = JEPAChunkCIFAR10(cifar_test,  train=False, pool_seed=1)
train_loader      = DataLoader(train_ds,      batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
train_eval_loader = DataLoader(train_eval_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
test_loader       = DataLoader(test_ds,       batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f"train={len(train_ds)}  test={len(test_ds)}")


## 3. Models — PBB encoder + dense predictor + EMA target + DINO centering

In [ ]:
context_encoder = PBBEncoder(
    d_model=D_MODEL, n_layers=N_LAYERS_ENC,
    n_heads=N_HEADS, dim_head=DIM_HEAD,
    block_size=BLOCK_SIZE, window=WINDOW,
    n_random=N_RANDOM, n_global=N_GLOBAL,
    signal_dim=3, coord_dim=2,
    fourier_dim=FOURIER_DIM, fourier_scale=FOURIER_SCALE,
).to(DEVICE)
target_encoder = copy.deepcopy(context_encoder).to(DEVICE)
for p in target_encoder.parameters(): p.requires_grad_(False)

predictor = PBBPredictor(
    d_model=D_MODEL, d_pred=D_PRED,
    n_layers=N_LAYERS_PRED, n_heads=N_HEADS_PRED, dim_head=DIM_HEAD_PRED,
    coord_dim=2, fourier_dim=FOURIER_DIM, fourier_scale=FOURIER_SCALE,
    pos_symmetric=True,
).to(DEVICE)
target_center = TargetCenter(D_MODEL, momentum=0.9).to(DEVICE)

if USE_DP:
    device_ids = list(range(N_GPUS))
    context_encoder = nn.DataParallel(context_encoder, device_ids=device_ids)
    target_encoder  = nn.DataParallel(target_encoder,  device_ids=device_ids)
    predictor       = nn.DataParallel(predictor,       device_ids=device_ids)

def _unwrap(m): return m.module if isinstance(m, nn.DataParallel) else m
print(f"context_encoder: {short_params(_unwrap(context_encoder))}")
print(f"predictor      : {short_params(_unwrap(predictor))}")


## 4. Helpers — compute SFC orderings per batch

In [ ]:
def quantize_coords(coords, side=IMAGE_SIZE, value_range=(-1.0, 1.0)):
    lo, hi = value_range
    norm = (coords - lo) / (hi - lo)
    norm = norm.clamp(0.0, 1.0)
    cell = (norm * (side - 1)).round().long().clamp(0, side - 1)
    return cell[..., 0] * side + cell[..., 1]


def orderings_from_coords(coords):
    # coords: (B, N, 2) → dict of perm/inverse per curve name
    grid_ranks = {k: v.to(coords.device) for k, v in GRID_RANKS.items()}
    cell_ids = quantize_coords(coords)
    out = {}
    for name, full in grid_ranks.items():
        ranks = full[cell_ids]                   # (B, N)
        perm = ranks.argsort(dim=-1)
        inv  = torch.empty_like(perm)
        arange = torch.arange(perm.shape[1], device=perm.device).unsqueeze(0).expand_as(perm)
        inv.scatter_(1, perm, arange)
        out[name] = {"perm": perm, "inverse": inv}
    return out


## 5. Optim + resume

In [ ]:
params = list(_unwrap(context_encoder).parameters()) + list(_unwrap(predictor).parameters())
optimizer = AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = EPOCHS * len(train_loader)
warmup_steps = WARMUP_EPOCHS * len(train_loader)
def lr_lambda(step):
    if step < warmup_steps:
        return step / max(warmup_steps, 1)
    p = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return 0.5 * (1 + math.cos(math.pi * p))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
momentum_gen = make_momentum_schedule(EMA_START, EMA_END, total_steps)

LAST = os.path.join(CKPT_DIR, "pbb_last.pt")
BEST = os.path.join(CKPT_DIR, "pbb_best.pt")
RESUME = True
history = {"loss": [], "diag_log": [], "diag_steps": [], "probe_accs": []}
start_epoch, best_loss, global_step = 1, float("inf"), 0
m = EMA_START
if RESUME and os.path.exists(LAST):
    s = torch.load(LAST, map_location=DEVICE, weights_only=False)
    _unwrap(context_encoder).load_state_dict(s["context_encoder"])
    _unwrap(target_encoder).load_state_dict(s["target_encoder"])
    _unwrap(predictor).load_state_dict(s["predictor"])
    target_center.load_state_dict(s["center"])
    optimizer.load_state_dict(s["opt"]); scheduler.load_state_dict(s["sched"])
    history = s.get("history", history)
    global_step = s.get("global_step", 0); best_loss = s.get("best_loss", float("inf"))
    start_epoch = s["epoch"] + 1
    for _ in range(global_step):
        try: m = next(momentum_gen)
        except StopIteration: m = EMA_END
    print(f"resumed @ ep {s['epoch']}, step {global_step}")
else:
    print("starting fresh.")


## 6. Probe — frozen encoder + linear head on mean-pooled context features

In [ ]:
class LinearProbe(nn.Module):
    def __init__(self, d, n=10):
        super().__init__()
        self.fc = nn.Linear(d, n)
    def forward(self, z): return self.fc(z)

def _ctx_features(batch, enc):
    # Forward encoder on pool, gather at ctx positions, mean-pool over ctx pixels.
    pool_coords = batch["pool_coords"].to(DEVICE)
    pool_signal = batch["pool_signal"].to(DEVICE)
    ctx_pool_pos = batch["ctx_pool_pos"].to(DEVICE)
    pool_ords = orderings_from_coords(pool_coords)
    with torch.no_grad():
        g_pool = enc(pool_signal, pool_coords, pool_ords)        # (B, K_pool, D)
    B, K_pool, D = g_pool.shape
    idx = ctx_pool_pos.unsqueeze(-1).expand(B, ctx_pool_pos.size(1), D)
    g_ctx = torch.gather(g_pool, 1, idx)                          # (B, K_ctx, D)
    return g_ctx.mean(dim=1)


def quick_probe(num_epochs=PROBE_EPOCHS, lr=1e-3, wd=1e-4):
    enc = _unwrap(context_encoder)
    saved_rg = [p.requires_grad for p in enc.parameters()]
    for p in enc.parameters(): p.requires_grad_(False)
    enc.eval()
    probe = LinearProbe(D_MODEL, 10).to(DEVICE)
    opt = AdamW(probe.parameters(), lr=lr, weight_decay=wd)
    ce  = nn.CrossEntropyLoss()
    for _ in range(num_epochs):
        probe.train()
        for b in train_eval_loader:
            y = b["label"].to(DEVICE)
            opt.zero_grad(set_to_none=True)
            ce(probe(_ctx_features(b, enc)), y).backward()
            opt.step()
    probe.eval()
    correct = total = 0
    with torch.no_grad():
        for b in test_loader:
            y = b["label"].to(DEVICE)
            correct += (probe(_ctx_features(b, enc)).argmax(-1) == y).sum().item()
            total += y.size(0)
    for p, rg in zip(enc.parameters(), saved_rg): p.requires_grad_(rg)
    return correct / max(total, 1)


## 7. Training loop — direct per-pixel target gather

In [ ]:
epoch = start_epoch - 1
try:
    for epoch in range(start_epoch, EPOCHS + 1):
        context_encoder.train(); predictor.train()
        epoch_loss, steps = 0.0, 0
        t0 = time.time()
        for batch in train_loader:
            pool_coords = batch["pool_coords"].to(DEVICE)            # (B, K_pool, 2)
            pool_signal = batch["pool_signal"].to(DEVICE)            # (B, K_pool, 3)
            tgt_pool_pos = batch["tgt_pool_pos"].to(DEVICE)          # (B, N_tgt)
            ctx_pool_pos = batch["ctx_pool_pos"].to(DEVICE)          # (B, K_ctx)

            pool_ords = orderings_from_coords(pool_coords)

            # ── Target: encoder on FULL pool, gather at target positions ──
            with torch.no_grad():
                g_tgt = target_encoder(pool_signal, pool_coords, pool_ords)
                                                                       # (B, K_pool, D)
                h_tgt_raw = gather_target_features(g_tgt, tgt_pool_pos)
                                                                       # (B, N_tgt, D)
                target_center.update(h_tgt_raw)
                h_tgt = F.layer_norm(target_center(h_tgt_raw), (h_tgt_raw.size(-1),))

            # ── Context: gather pool_signal/coords at ctx positions, encode ──
            B, _, D = pool_signal.shape
            ctx_signal = torch.gather(pool_signal, 1,
                                       ctx_pool_pos.unsqueeze(-1).expand(B, ctx_pool_pos.size(1), 3))
            ctx_coords = torch.gather(pool_coords, 1,
                                       ctx_pool_pos.unsqueeze(-1).expand(B, ctx_pool_pos.size(1), 2))
            ctx_ords = orderings_from_coords(ctx_coords)
            g_ctx = context_encoder(ctx_signal, ctx_coords, ctx_ords)  # (B, K_ctx, D)

            # ── Predictor: ctx tokens + mask tokens at target coords ──
            tgt_coords = torch.gather(pool_coords, 1,
                                       tgt_pool_pos.unsqueeze(-1).expand(B, tgt_pool_pos.size(1), 2))
            h_pred = predictor(g_ctx, tgt_coords, ctx_coords=ctx_coords)

            loss = jepa_loss(h_pred, h_tgt, loss_type="smooth_l1")
            optimizer.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            optimizer.step(); scheduler.step()

            try: m = next(momentum_gen)
            except StopIteration: m = EMA_END
            ema_update(_unwrap(target_encoder), _unwrap(context_encoder), m)

            global_step += 1; epoch_loss += loss.item(); steps += 1
            if global_step % LOG_EVERY == 0:
                pred_std = h_pred.std(dim=0).mean().item()
                tgt_std  = h_tgt.std(dim=0).mean().item()
                cos = F.cosine_similarity(h_pred, h_tgt, dim=-1).mean().item()
                print(f"[ep{epoch:03d} st{global_step:06d}]  loss={loss.item():.4f}  "
                      f"pred_std={pred_std:.3f}  tgt_std={tgt_std:.3f}  cos={cos:.3f}  "
                      f"lr={scheduler.get_last_lr()[0]:.1e}  m={m:.5f}")
                history["diag_steps"].append(global_step)
                history["diag_log"].append({"pred_std": pred_std, "tgt_std": tgt_std,
                                              "cos": cos, "loss": loss.item()})

        avg = epoch_loss / max(steps, 1)
        history["loss"].append(avg)
        improved = avg < best_loss
        if improved: best_loss = avg
        state = {
            "epoch": epoch,
            "context_encoder": _unwrap(context_encoder).state_dict(),
            "target_encoder":  _unwrap(target_encoder).state_dict(),
            "predictor":       _unwrap(predictor).state_dict(),
            "center":          target_center.state_dict(),
            "opt": optimizer.state_dict(), "sched": scheduler.state_dict(),
            "global_step": global_step, "best_loss": best_loss, "history": history,
        }
        torch.save(state, LAST)
        if improved: torch.save(state, BEST)
        print(f"=== ep {epoch:03d}/{EPOCHS}  loss={avg:.4f}  m={m:.5f}  "
              f"{time.time()-t0:.1f}s" + ("  *" if improved else ""))
        if PROBE_INTERVAL > 0 and epoch % PROBE_INTERVAL == 0:
            tp = time.time()
            acc = quick_probe()
            history["probe_accs"].append((epoch, acc))
            print(f"  [probe ep {epoch:03d}]  test_acc = {acc:.4f}  ({time.time()-tp:.1f}s)")
    print("\nDone.")
except KeyboardInterrupt:
    print(f"\nInterrupted at epoch {epoch}.  Checkpoints in {CKPT_DIR}.")


## 8. Final long-form linear probe

In [ ]:
LOAD_BEST = True
n_probe_epochs = 30
probe_lr  = 1e-3
probe_wd  = 1e-4

if LOAD_BEST and os.path.exists(BEST):
    print(f"loading best checkpoint @ {BEST}")
    s = torch.load(BEST, map_location=DEVICE, weights_only=False)
    _unwrap(context_encoder).load_state_dict(s["context_encoder"])
    print(f"  ckpt epoch = {s['epoch']}, train_loss = {s.get('best_loss', 'n/a')}")
else:
    print("using current weights")

enc = _unwrap(context_encoder); enc.eval()
for p in enc.parameters(): p.requires_grad_(False)
probe = LinearProbe(D_MODEL, 10).to(DEVICE)
opt = AdamW(probe.parameters(), lr=probe_lr, weight_decay=probe_wd)
ce  = nn.CrossEntropyLoss()

history_probe = {"train_acc": [], "test_acc": []}
print(f"\ntraining probe for {n_probe_epochs} epochs...")
for ep in range(1, n_probe_epochs + 1):
    probe.train()
    correct = total = 0
    for b in train_eval_loader:
        y = b["label"].to(DEVICE)
        logits = probe(_ctx_features(b, enc))
        opt.zero_grad(set_to_none=True); ce(logits, y).backward(); opt.step()
        correct += (logits.argmax(-1) == y).sum().item(); total += y.size(0)
    train_acc = correct / total
    probe.eval()
    c2 = t2 = 0
    with torch.no_grad():
        for b in test_loader:
            y = b["label"].to(DEVICE)
            c2 += (probe(_ctx_features(b, enc)).argmax(-1) == y).sum().item(); t2 += y.size(0)
    test_acc = c2 / t2
    history_probe["train_acc"].append(train_acc); history_probe["test_acc"].append(test_acc)
    print(f"  ep {ep:02d}/{n_probe_epochs}  train_acc={train_acc:.4f}  test_acc={test_acc:.4f}")
best_test = max(history_probe["test_acc"])
print(f"\nfinal probe accuracy:  best test = {best_test:.4f}  (final = {history_probe['test_acc'][-1]:.4f})")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([a*100 for a in history_probe["train_acc"]], 'o-', label="train", color='C0')
ax.plot([a*100 for a in history_probe["test_acc"]],  's-', label="test",  color='C3')
ax.set_xlabel("probe epoch"); ax.set_ylabel("accuracy (%)")
ax.set_title(f"PBB-CIFAR-10 final linear probe — best test = {best_test*100:.2f}%")
ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()


## 9. Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
if history["loss"]:
    axes[0].plot(range(1, len(history["loss"])+1), history["loss"], lw=1.5)
    axes[0].set_xlabel("epoch"); axes[0].set_ylabel("smooth_L1"); axes[0].set_title("JEPA loss")
    axes[0].grid(alpha=0.3)
if history["diag_log"]:
    steps = history["diag_steps"]
    axes[1].plot(steps, [d["pred_std"] for d in history["diag_log"]], label="pred_std", color="C0")
    axes[1].plot(steps, [d["tgt_std"]  for d in history["diag_log"]], label="tgt_std",  color="C2")
    axes[1].plot(steps, [d["cos"]      for d in history["diag_log"]], label="cos",      color="C3")
    axes[1].set_xlabel("step"); axes[1].set_title("diagnostics"); axes[1].legend(fontsize=8)
    axes[1].grid(alpha=0.3)
if history.get("probe_accs"):
    eps, accs = zip(*history["probe_accs"])
    axes[2].plot(eps, [a*100 for a in accs], 'o-', color='C3')
    axes[2].set_ylim(0, 100); axes[2].set_xlabel("epoch"); axes[2].set_ylabel("probe acc (%)")
    axes[2].set_title(f"during-training probe (best = {max(accs)*100:.2f}%)"); axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()
